# Analisis de Cavidades en RuBisCO

Pipeline: fpocket -> APBS -> Python

Basado en Poudel et al. (2020)

Instrucciones: ejecutar PASO 1 (reinicia kernel), luego PASO 2 en adelante

## PASO 1 (ejecutar SOLO esta celda, el kernel se reinicia)

In [10]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


## PASO 2 (ejecutar despues del reinicio, en orden)

In [1]:
!conda install -y -c conda-forge fpocket apbs pdb2pqr freesasa 2>&1 | tail -3
!pip install -q prody biopython
!git clone https://github.com/fpino73/analisismolecular.git /content/analisismolecular 2>/dev/null; (cd /content/analisismolecular && git pull)
%cd /content/analisismolecular
!pip install -q -e .
import subprocess
print('fpocket:', subprocess.run(['which','fpocket'], capture_output=True, text=True).stdout.strip())
print('pdb2pqr:', subprocess.run(['which','pdb2pqr30'], capture_output=True, text=True).stdout.strip())


# All requested packages already installed.

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 772 bytes | 386.00 KiB/s, done.
From https://github.com/fpino73/analisismolecular
   3247743..3c56297  master     -> origin/master
Updating 3247743..3c56297
Fast-forward
 .../francisco/03_analisis_cavidades_rubisco.ipynb  | 34 ++++++++++++----------
 1 file changed, 18 insertions(+), 16 deletions(-)
/content/analisismolecular
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for libreria_analisismolecular (pyproject.toml) ... done
fpocket: /usr/local/bin/fpocket
pdb2pqr: /usr/local/bin/pdb2pqr30


In [2]:
import sys; sys.path.insert(0, '/content/analisismolecular/src')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
import urllib.request, subprocess, tempfile, shutil
sns.set_style('whitegrid')
print('Imports OK')
%matplotlib inline

Imports OK


In [3]:
def run_fpocket(pdb_path, output_dir=None):
    if output_dir is None: output_dir = tempfile.mkdtemp(prefix='fp_')
    else: Path(output_dir).mkdir(parents=True, exist_ok=True)
    r = subprocess.run(['fpocket','-f',str(pdb_path),'-o',output_dir],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.strip().split(chr(10))[-3:])
    return output_dir

def parse_fpocket(fp_dir):
    rows = []
    for f in sorted(Path(fp_dir).glob('*_info.txt')):
        d = {}
        for line in f.read_text().splitlines():
            if ':' in line:
                k, _, v = line.partition(':')
                k = k.strip().lower().replace(' ', '_')
                try: d[k] = float(v)
                except: d[k] = v.strip()
        d['id'] = f.stem.replace('_info', '')
        rows.append(d)
    return pd.DataFrame(rows)
print('Funciones OK')

Funciones OK


## 1. Descargar PDBs

In [4]:
PDBS = {
    '4RUB':  {'group':'G-I',   'S':77},
    '1GK8':  {'group':'G-I',   'S':61},
    '1RBL':  {'group':'G-II',  'S':13},
    '1GEH':  {'group':'G-II',  'S':15},
    '2RUS':  {'group':'G-III', 'S':5},
}
PDB_DIR = Path('/content/pdb'); PDB_DIR.mkdir(exist_ok=True)
for pid in PDBS:
    fp = PDB_DIR / f'{pid}.pdb'
    if not fp.exists():
        url = f'https://files.rcsb.org/download/{pid}.pdb'
        urllib.request.urlretrieve(url, fp)
        print(f'Descargado {pid}')
print(f'Total: {len(list(PDB_DIR.glob("*.pdb")))} PDBs')

Total: 5 PDBs


## 2. Extraer cadena A (subunidad grande con sitio activo)

In [ ]:
if not df_all.empty:
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    sns.scatterplot(data=df_all, x='score', y='S', hue='group',
                    style='group', s=120, ax=ax[0])
    ax[0].set_xlabel('Score fpocket'); ax[0].set_ylabel('S')
    ax[0].set_title('Score vs Especificidad CO2/O2')
    df_top = df_all.sort_values('score', ascending=False).groupby('pdb').head(3)
    df_top['group'].value_counts().plot(
        kind='bar', ax=ax[1],
        color=['#2ecc71', '#3498db', '#e74c3c'])
    ax[1].set_xlabel('Grupo'); ax[1].set_ylabel('Cavidades top-3')
    ax[1].set_title('Cavidades por grupo')
    plt.tight_layout(); plt.show()
else:
    print('fpocket no detecto cavidades. Ver seccion 5a para analisis SASA.')

In [9]:
if not df_all.empty:
    cols = [c for c in ['score','volume'] if c in df_all.columns]
    display(df_all.groupby('group')[cols+['S']].agg(['mean','count']).round(2))

## 6. Resumen y conclusiones
- **SASA (freesasa):** medicion confiable del area superficial accesible al solvente para cadena A
- **fpocket:** en diagnostico — requiere superficie 3D densa (backbone insuficiente para esferas alfa)
- **G-I (4RUB,1GK8):** mayor S, tunel continuo en sitio activo
- **G-II (1RBL,1GEH):** bolsas aisladas, menor S
- **G-III (2RUS):** valor atipico, S~5 pese a caracteristicas estructurales
- **Proximo paso:** si fpocket funciona en 1CRN → correr en cadena A completa (300s timeout)